# ノートブック② RoboGen フルパイプライン

[RoboGen](https://github.com/Genesis-Embodied-AI/RoboGen)（LLM によるロボットタスク自動生成 + スキル学習）を Google Colab 上で実行します。

## 実行ステップ
| ステップ | 内容 | OpenAI キー |
|---|---|---|
| **A. 再生** | 例題タスクのシーンを構築し周回カメラ動画を生成（環境構築成功の関門） | 不要 |
| **B. タスク生成** | タスク説明文から LLM がタスク config・報酬関数を自動生成 | **必要** |
| **C. スキル学習** | モーションプランニング + 強化学習 (SB3) でスキルを学習 | 不要 |

## 前提
- **GPU ランタイム必須**（ランタイム → ランタイムのタイプを変更 → T4 GPU）
- ステップB には OpenAI API キー（Colab の Secrets に `OPENAI_API_KEY` として登録）
- 所要時間: セットアップ ~10分 + ステップA ~10分 + B ~10分 + C は学習量次第（短縮設定で ~30分）

## 本ノートブックでの RoboGen の変更点（詳細は `patches/README.md`）
- 公開版 RoboGen は Python 3.9 / ray 1.13 / 旧 openai SDK 前提のため、Colab (Python 3.12) 向けに
  パッチを適用します: RL は stable-baselines3 に置き換え、gym は gymnasium への互換 shim で動かします。

上から順にセルを実行してください。

In [ ]:
# === セル2: GPU 確認 ===
!nvidia-smi

import torch                       # Colab プリインストールの PyTorch
assert torch.cuda.is_available(), (
    "CUDA GPU が見つかりません。「ランタイム」→「ランタイムのタイプを変更」で GPU を選択してください。"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# === セル3: 本リポジトリの clone ===
import os                          # パス・環境変数操作用

REPO_URL = "https://github.com/ringo7pie/RoboGen-GenesisWorld.git"   # 本リポジトリの公開 URL
REPO_DIR = "/content/RoboGen-GenesisWorld"     # 本リポジトリの clone 先
ROBOGEN_DIR = "/content/RoboGen"               # RoboGen 本体の clone 先（セル4で取得）
os.environ["ROBOGEN_DIR"] = ROBOGEN_DIR        # 各スクリプトが参照する

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("既に clone 済みのためスキップします。")

In [ ]:
# === セル4: RoboGen セットアップ一括実行（5〜10分） ===
# RoboGen の clone（コミット固定）→ 依存インストール → 置き換えモジュール配置 → パッチ適用
!bash {REPO_DIR}/scripts/setup_robogen.sh

In [ ]:
# === セル5: セルフチェック ===
# import・パッチ適用・ヘッドレス描画・モーションプランニングを検査する。
# NG がある場合は、このセルの出力全体をエラー報告として共有してください。
!cd {ROBOGEN_DIR} && python {REPO_DIR}/scripts/check_env.py --stage robogen

In [ ]:
# === セル6: Google Drive マウント + アセット取得 ===
# PartNet-Mobility（パース済み版、RoboGen 公式配布）を自動ダウンロードします。
# Drive をマウントすると zip がキャッシュされ、次回セッションの再ダウンロードを回避できます。
# （Drive を使いたくない場合は mount の行をコメントアウトしても動作します）
from google.colab import drive     # Drive マウント用
drive.mount("/content/drive")

!cd {ROBOGEN_DIR} && python {REPO_DIR}/scripts/download_assets.py --download --verify

## ステップA: 例題タスクの再生（OpenAI キー不要）

RoboGen 同梱の例題タスク（ノート PC を開く）のシーンを構築し、周回カメラで撮影した動画を生成します。
**このセルが通れば環境構築は成功です。**

In [ ]:
# === セル7: シーン再構築 + 動画化 ===
TASK_CONFIG = "example_tasks/Open_Laptop/Open_Laptop_The_robotic_arm_opens_the_unfolded_state_of_the_laptops_screen.yaml"   # 例題タスクの定義 YAML

!cd {ROBOGEN_DIR} && python {REPO_DIR}/scripts/run_robogen_task.py --mode replay --task-config {TASK_CONFIG}

# --- 生成された mp4（最新の construction.mp4）をインライン表示する ---
import glob                        # 出力動画の探索用
from IPython.display import Video  # インライン再生用

mp4s = sorted(glob.glob(f"{ROBOGEN_DIR}/example_tasks/**/construction.mp4", recursive=True),
              key=os.path.getmtime)             # 更新時刻順の mp4 一覧
assert mp4s, "再生動画が生成されていません。上のセルのログを確認してください。"
print(f"再生動画: {mp4s[-1]}")
Video(mp4s[-1], embed=True, width=640)

## ステップB: LLM によるタスク新規生成（OpenAI キー必要）

事前準備: Colab 左側のカギアイコン（Secrets）で `OPENAI_API_KEY` を登録し、
「ノートブックからのアクセス」を ON にしてください。

In [ ]:
# === セル8: OpenAI キー設定 + 疎通確認 ===
from google.colab import userdata  # Colab Secrets の読み取り用

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")   # 環境変数経由で RoboGen に渡す
except Exception as e:
    raise RuntimeError(
        "Colab Secrets から OPENAI_API_KEY を取得できません。"
        "左側のカギアイコン → 新しいシークレットで OPENAI_API_KEY を登録し、アクセスを ON にしてください。"
    ) from e

os.environ["ROBOGEN_LLM_MODEL"] = "gpt-4o"     # タスク生成に使うモデル（gpt-4 は廃止予定のため gpt-4o を既定に）

# --- 軽い呼び出しを1回だけ行って疎通を確認する（課金は最小限） ---
from openai import OpenAI          # 新 openai SDK
client = OpenAI()                  # OPENAI_API_KEY を自動で読む
res = client.chat.completions.create(model=os.environ["ROBOGEN_LLM_MODEL"],
                                     messages=[{"role": "user", "content": "ping"}], max_tokens=5)
print(f"疎通 OK: model={os.environ['ROBOGEN_LLM_MODEL']}")
print("注意: ステップB のタスク生成は GPT への複数回の問い合わせを行うため、API 課金が発生します。")

In [ ]:
# === セル9: タスク生成の実行 ===
# 前提: OpenAI API に課金クレジットが必要です（ChatGPT とは別契約。platform.openai.com の
#       Billing でチャージ）。未チャージだと insufficient_quota エラーになります。
# タスク説明文と対象オブジェクト（PartNet-Mobility のカテゴリ名と ID）を指定して、
# タスク config・サブステップ分解・報酬関数コードを LLM に生成させます。
# 追加オブジェクトの検索に Objaverse 埋め込みが必要になる場合があるため、先に取得します
# （数十 GB 規模。ディスク空き容量が不足する場合は自動で中止されます）。
!cd {ROBOGEN_DIR} && python {REPO_DIR}/scripts/download_assets.py --download --with-embeddings

TASK_DESCRIPTION = "open the door of the storage furniture"   # 生成したいタスクの説明文（英語）
TASK_OBJECT = "StorageFurniture"                              # 対象の PartNet-Mobility カテゴリ
TASK_OBJECT_ID = "45374"                                      # 対象オブジェクトの PartNet ID

# -m 実行にすることで cwd（RoboGen ルート）が import パスに入り、gpt_4 パッケージが解決される
!cd {ROBOGEN_DIR} && python -m gpt_4.prompts.prompt_from_description \
    --task_description "{TASK_DESCRIPTION}" --object {TASK_OBJECT} --object_path {TASK_OBJECT_ID}

# --- 生成されたタスク config を確認する ---
generated = sorted(glob.glob(f"{ROBOGEN_DIR}/data/generated_task_from_description/**/*.yaml", recursive=True),
                   key=os.path.getmtime)       # 生成された YAML 一覧（更新時刻順）
assert generated, "タスク config が生成されていません。上のログを確認してください。"
GENERATED_CONFIG = generated[-1]               # 最新の生成タスク config
print(f"生成されたタスク config: {GENERATED_CONFIG}\n" + "=" * 60)
print(open(GENERATED_CONFIG).read())

## ステップC: スキル学習

サブステップごとに、モーションプランニング（把持など）と強化学習（SB3 SAC、操作など）でスキルを学習します。

- 既定はデモ用の**短縮設定（2万ステップ/サブステップ、~30分）**です。動作確認には十分ですが、
  タスク成功まで学習するには `--timesteps 1000000` 程度が必要です（Colab のセッション上限に注意）。
- 対象タスクは既定で**ステップA の例題タスク**です。ステップB の生成タスクを学習する場合は
  `target` を `GENERATED_CONFIG` に変更してください。

In [ ]:
# === セル10: スキル学習の実行（短縮設定で ~30分） ===
target = TASK_CONFIG               # 学習対象のタスク config（生成タスクなら GENERATED_CONFIG に変更）

!cd {ROBOGEN_DIR} && python {REPO_DIR}/scripts/run_robogen_task.py --mode learn \
    --task-config "{target}" --timesteps 20000 --eval-interval 5000

In [ ]:
# === セル11: 結果の表示 + Google Drive への保存 ===
import shutil                      # ファイルコピー用
import datetime                    # 保存フォルダ名用

# --- 学習結果の動画（最新の mp4）を表示する ---
from IPython.display import Video  # インライン再生用
result_mp4s = sorted(glob.glob(f"{ROBOGEN_DIR}/example_tasks/**/*.mp4", recursive=True) +
                     glob.glob(f"{ROBOGEN_DIR}/data/**/*.mp4", recursive=True),
                     key=os.path.getmtime)     # 全出力 mp4（更新時刻順）
assert result_mp4s, "結果動画が見つかりません。"
latest_mp4 = result_mp4s[-1]                   # 最新の結果動画
print(f"結果動画: {latest_mp4}")

# --- Drive へ動画・ログ一式を保存する（マウント済みの場合） ---
if os.path.isdir("/content/drive/MyDrive"):
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")          # 保存フォルダの識別子
    save_dir = f"/content/drive/MyDrive/robogen_outputs/{stamp}"       # Drive 上の保存先
    os.makedirs(save_dir, exist_ok=True)
    for f in result_mp4s[-5:]:                 # 直近の動画 5 件を保存
        shutil.copy(f, save_dir)
    print(f"Drive に保存しました: {save_dir}")
else:
    print("Drive 未マウントのため保存をスキップしました（セル6を実行すると保存できます）。")

Video(latest_mp4, embed=True, width=640)

## 次の一歩

- **別の例題タスクを試す**: セル7の `TASK_CONFIG` を `example_tasks/` 配下の別タスクの YAML に変更
  （46 種類の例題があります: `!ls /content/RoboGen/example_tasks/`）
- **本格的に学習する**: セル10 の `--timesteps` を増やす（例 `1000000`）。Colab のセッション切れに備え、
  途中結果は `save_path` 配下の `latest_model.zip` / `best_model.zip` に保存されています
- **生成タスクの品質を上げる**: セル9 の `TASK_DESCRIPTION` を具体的に書く、`ROBOGEN_LLM_MODEL` を変更する
- **うまく動かないとき**: `docs/troubleshooting.md` を参照。それでも解決しなければ
  セル5（セルフチェック）の出力とエラーログを添えて報告してください